In [14]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Qwen/Qwen2.5-0.5B-Instruct",
    local_dir="D:/huggingface_models/Qwen2.5-0.5B"
)

Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 521.10it/s]


'D:\\huggingface_models\\Qwen2.5-0.5B'

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "D:/huggingface_models/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

print("Model loaded successfully!")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 302.31it/s]

Model loaded successfully!


In [16]:
from langgraph.graph import StateGraph,END,START
from typing import TypedDict


In [5]:
# create a state 

class LLMState(TypedDict):
    question:str
    answer:str
    

In [17]:
def llm_qa(state: LLMState) -> LLMState:
    # Extract the question from the LangGraph state
    question = state['question']

    # Build a chat prompt for Qwen2.5-Instruct
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Give clear and accurate answers."},
        {"role": "user", "content": question}
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    # Generate the answer with the local Transformers model
    outputs = model.generate(
        input_ids,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode only newly generated tokens, not the original prompt
    new_tokens = outputs[0][input_ids.shape[-1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Return the updated state
    return {**state, "answer": answer}


In [18]:
# create a graph
graph = StateGraph(LLMState)

#  add nodes
graph.add_node('llm_qa',llm_qa)

# add edge
graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

# compile
workflow = graph.compile()

In [20]:
workflow = graph.compile()

final_state = workflow.invoke({
    'question': 'How far is the Moon from Earth?'
})

print(final_state['answer'])

AttributeError: 